In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format("parquet")\
    .load("abfss://bronze@databricksetestorage.dfs.core.windows.net/customers")
df.display()

### **Data Enrichment**

In [0]:
df=df.drop("_rescued_data")
df.display()

In [0]:
df=df.withColumn("domains",split(col("email"),"@")[1])
df.display()

In [0]:
df.groupBy("domains").agg(count("customer_id").alias("total_customers")).sort("total_customers",ascending=False).display()

In [0]:
df_gmail=df.filter(col("domains")=="gmail.com")
df_gmail.display()

df_yahoo=df.filter(col("domains")=="yahoo.com")
df_yahoo.display()

df_hotmail=df.filter(col("domains")=="hotmail.com")
df_hotmail.display()

In [0]:
df=df.withColumn("fullname",concat("first_name",lit(" "),"last_name"))
df=df.drop("first_name","last_name")
df.display()

### **Data Writing**

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@databricksetestorage.dfs.core.windows.net/customers")

In [0]:
%sql
create table if not exists databricks_cat.silver.customers_silver
using delta
location "abfss://silver@databricksetestorage.dfs.core.windows.net/customers"